# Hopf Coassociativity PoC on sDTM

Tests whether a coproduct coassociativity auxiliary loss produces signal on sDTM's Active↔Logical task.

Three conditions: **baseline** (no aux loss), **coprod** (coassociativity loss), **random** (random projection control).

Set `QUICK_MODE = True` for a fast debug run (2000 steps). Set to `False` for the real experiment (20000 steps).

In [ ]:
# Cell 1: Setup
QUICK_MODE = True  # Set False for full 20k-step runs
USE_AMP = True     # Automatic Mixed Precision (2-3x speedup on T4)

import os, subprocess, sys

# Clone sDTM if not present
if not os.path.exists('sdtm'):
    subprocess.run(['git', 'clone', 'https://github.com/psoulos/sdtm.git'], check=True)

# Install deps (quiet)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'torch_geometric', 'wandb', 'nltk', 'numpy', 'matplotlib'],
               check=True)

os.chdir('sdtm')
sys.path.insert(0, '.')
os.environ['WANDB_MODE'] = 'disabled'

# Init wandb in disabled mode so wandb.log() calls don't crash
import wandb
wandb.init()

# --- Patch models.py ---
with open('models.py', 'r') as f:
    src = f.read()
patched = src
# PyTorch >= 2.4: .view -> .reshape for non-contiguous tensors
patched = patched.replace('out.view(num_trees, -1)', 'out.reshape(num_trees, -1)')
patched = patched.replace('out.view(bsz, -1)', 'out.reshape(bsz, -1)')
patched = patched.replace('.view(num_nodes, -1)', '.reshape(num_nodes, -1)')
patched = patched.replace('tree_encoding.view(bsz, 1, -1)', 'tree_encoding.reshape(bsz, 1, -1)')
if patched != src:
    with open('models.py', 'w') as f:
        f.write(patched)
    print('Patched models.py for PyTorch compatibility')

# --- Patch data.py: add persistent_workers + prefetch_factor ---
with open('data.py', 'r') as f:
    dsrc = f.read()
if 'persistent_workers' not in dsrc:
    import re
    dsrc = re.sub(
        r'pin_memory=True, collate_fn=_sparse_collate_fn\b',
        'pin_memory=True, collate_fn=_sparse_collate_fn, '
        'persistent_workers=True if num_workers > 0 else False, '
        'prefetch_factor=4 if num_workers > 0 else None',
        dsrc,
    )
    with open('data.py', 'w') as f:
        f.write(dsrc)
    print('Patched data.py: persistent_workers + prefetch_factor')

print('Setup complete. CWD:', os.getcwd())

In [ ]:
# Cell 2: Imports and compatibility patches
import math, random, time, copy
from collections import defaultdict

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# Patch CosineAnnealingLR for PyTorch >= 2.4 (removed 'verbose' kwarg)
_orig_cosine_init = torch.optim.lr_scheduler.CosineAnnealingLR.__init__
def _patched_cosine_init(self, optimizer, T_max, eta_min=0, last_epoch=-1, **kwargs):
    kwargs.pop('verbose', None)
    _orig_cosine_init(self, optimizer, T_max, eta_min=eta_min, last_epoch=last_epoch, **kwargs)
torch.optim.lr_scheduler.CosineAnnealingLR.__init__ = _patched_cosine_init

from config import parse_args
from TPR_utils import TPR, decoded_tpr_to_tree_fn, SparseTPR
from models import DiffTreeMachine
from trainer import Trainer
import data

print('Imports OK')
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 3: Standalone car/cdr for SparseTPR
#
# SparseTPR stores:
#   _indices: shape [2, nnz] — row 0 = batch_idx, row 1 = role_idx (Gorn address)
#   _values:  shape [nnz, d_filler] — continuous filler vectors
#   NOTE: indices may be int32, so we cast to long where needed.
#
# Gorn addressing (1-indexed binary tree):
#   root = 1, left child = 2*parent, right child = 2*parent + 1
#   car = left subtree (even children), cdr = right subtree (odd children, excl root)
#   After extracting a subtree, shift roles right by 1 bit (parent = child >> 1)

def sparse_tpr_car(spt):
    """Extract left subtree: keep even-role entries, shift roles >> 1."""
    roles = spt.indices()[1]
    mask = (roles % 2 == 0) & (roles >= 2)  # even roles, excluding 0
    new_roles = roles[mask] >> 1
    new_indices = torch.stack([spt.indices()[0][mask], new_roles])
    new_values = spt.values()[mask]
    return SparseTPR(new_indices, new_values)


def sparse_tpr_cdr(spt):
    """Extract right subtree: keep odd-role entries (excl root=1), shift roles >> 1."""
    roles = spt.indices()[1]
    mask = (roles % 2 == 1) & (roles != 1)  # odd roles, excluding root
    new_roles = roles[mask] >> 1
    new_indices = torch.stack([spt.indices()[0][mask], new_roles])
    new_values = spt.values()[mask]
    return SparseTPR(new_indices, new_values)


def tree_max_depth_per_batch(spt, bsz):
    """Compute max tree depth per batch element via floor(log2(max_role))."""
    roles = spt.indices()[1].float()
    batch_idx = spt.indices()[0].long()
    max_roles = torch.zeros(bsz, device=roles.device)
    max_roles.scatter_reduce_(0, batch_idx, roles, reduce='amax')
    return torch.floor(torch.log2(max_roles.clamp(min=1))).long()


def sum_fillers_per_batch(spt, bsz, d_filler):
    """Sum filler vectors per batch element. Returns (bsz, d_filler)."""
    result = torch.zeros(bsz, d_filler, device=spt.values().device)
    batch_idx = spt.indices()[0].long()
    result.scatter_add_(0, batch_idx.unsqueeze(1).expand(-1, d_filler), spt.values())
    return result


print('car/cdr utilities defined')

In [ ]:
# Cell 4: Coassociativity loss
#
# For a tree T, the coproduct coassociativity axiom states:
#   (Delta ⊗ id) ∘ Delta = (id ⊗ Delta) ∘ Delta
#
# Using car/cdr as the coproduct components and sum-of-fillers as the
# permutation-invariant encoder f:
#   Left:  f(car(car(T))) + f(cdr(car(T))) + f(cdr(T))
#   Right: f(car(T))      + f(car(cdr(T))) + f(cdr(cdr(T)))
#
# Loss = mean over batch of ||left - right||^2, restricted to trees with depth >= 2.
# GPU-optimized: no .any() sync points; uses masked multiplication instead.

def coassociativity_loss(output_spt, bsz, d_filler):
    """Compute coassociativity violation on the output SparseTPR."""
    vals = output_spt.values()
    if vals.shape[0] == 0:
        return vals.sum() * 0.0  # zero that inherits device + graph

    # NaN/inf guard
    if not torch.isfinite(vals).all():
        return vals.sum() * 0.0

    depths = tree_max_depth_per_batch(output_spt, bsz)
    deep_mask = (depths >= 2).float()  # (bsz,) — no .any() sync

    car_T = sparse_tpr_car(output_spt)
    cdr_T = sparse_tpr_cdr(output_spt)

    f = lambda spt: sum_fillers_per_batch(spt, bsz, d_filler)

    # Left-first decomposition: (Delta ⊗ id) ∘ Delta
    left_first = f(sparse_tpr_car(car_T)) + f(sparse_tpr_cdr(car_T)) + f(cdr_T)

    # Right-first decomposition: (id ⊗ Delta) ∘ Delta
    right_first = f(car_T) + f(sparse_tpr_car(cdr_T)) + f(sparse_tpr_cdr(cdr_T))

    diff = left_first - right_first  # (bsz, d_filler)
    per_sample = (diff ** 2).sum(dim=1)  # (bsz,)
    masked = per_sample * deep_mask
    denom = deep_mask.sum().clamp(min=1.0)
    return masked.sum() / denom


print('Coassociativity loss defined (GPU-optimized, no sync points)')

In [ ]:
# Cell 5: Random projection control loss
#
# Control: apply two random frozen projections to the sum-of-fillers
# and penalize their difference. Same computational cost as coassociativity
# but no algebraic meaning.

class RandomProjectionLoss(nn.Module):
    def __init__(self, d_filler):
        super().__init__()
        # Frozen random matrices
        self.register_buffer('W1', torch.randn(d_filler, d_filler) / math.sqrt(d_filler))
        self.register_buffer('W2', torch.randn(d_filler, d_filler) / math.sqrt(d_filler))

    def forward(self, output_spt, bsz, d_filler):
        vals = output_spt.values()
        if vals.shape[0] == 0:
            return vals.sum() * 0.0
        if not torch.isfinite(vals).all():
            return vals.sum() * 0.0
        s = sum_fillers_per_batch(output_spt, bsz, d_filler)
        diff = s @ self.W1 - s @ self.W2
        return (diff ** 2).sum(dim=1).mean()


print('RandomProjectionLoss defined')

In [ ]:
# Cell 5b: Validation — discrete tree test + NaN guard test
#
# For fully discrete binary trees, coassociativity should be trivially zero
# because both decomposition orders sum the same multiset of nodes.
# This test confirms the math is correct.

def test_discrete_tree_coassociativity():
    """Verify coassociativity loss == 0 for a perfect discrete binary tree."""
    # Build a perfect binary tree of depth 3: nodes at roles 1,2,3,4,5,6,7
    # batch_size=1, d_filler=4, one-hot-like fillers
    bsz, d_filler = 1, 4
    roles = [1, 2, 3, 4, 5, 6, 7]
    indices = torch.tensor([[0]*len(roles), roles], dtype=torch.int32)
    # Each node gets a unique filler (simulates discrete tokens)
    values = torch.eye(d_filler)[torch.arange(len(roles)) % d_filler].clone()
    values.requires_grad_(True)

    spt = SparseTPR(indices, values)
    loss = coassociativity_loss(spt, bsz, d_filler)
    assert abs(loss.item()) < 1e-6, f"Discrete tree loss should be ~0, got {loss.item()}"
    print(f"  Discrete tree test PASSED (loss={loss.item():.2e})")

    # Also verify gradient flows
    loss_grad = coassociativity_loss(spt, bsz, d_filler)
    loss_grad.backward()
    assert values.grad is not None, "Gradient should flow through coassociativity loss"
    print(f"  Gradient flow test PASSED")

    # NaN input guard test
    nan_values = torch.full((3, d_filler), float('nan'))
    nan_spt = SparseTPR(torch.tensor([[0,0,0],[1,2,3]], dtype=torch.int32), nan_values)
    nan_loss = coassociativity_loss(nan_spt, 1, d_filler)
    assert nan_loss.item() == 0.0, f"NaN input should return 0, got {nan_loss.item()}"
    print(f"  NaN guard test PASSED")

test_discrete_tree_coassociativity()
print("All validation tests passed")

In [ ]:
# Cell 6: Monkey-patch process_batch + train_epoch for AMP and auxiliary loss
#
# process_batch returns a 9-tuple:
#   (loss, correct_tokens, token_total, correct_sequences, bsz,
#    debug_info, entropies, output, perplexity)
# output (index 7) is the SparseTPR we need for the auxiliary loss.
#
# AMP strategy: wrap the entire train_epoch forward+backward with autocast/GradScaler.
# We monkey-patch train_epoch to add AMP around the existing flow.

def patch_trainer(trainer, aux_loss_fn, aux_lambda, d_filler):
    """Monkey-patch trainer for auxiliary loss injection + AMP."""
    trainer._aux_loss_fn = aux_loss_fn
    trainer._aux_lambda = aux_lambda
    trainer._aux_d_filler = d_filler
    trainer._aux_history = []  # (step, aux_loss_value)
    trainer._train_history = []  # (step, train_loss_value)
    trainer._global_step_counter = [0]

    use_amp = USE_AMP and torch.cuda.is_available()
    trainer._amp_scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    trainer._amp_enabled = use_amp

    # --- Patch process_batch: inject auxiliary loss ---
    original_process_batch = trainer.process_batch

    def patched_process_batch(batch, **kwargs):
        result = list(original_process_batch(batch, **kwargs))
        loss = result[0]
        output_spt = result[7]

        step = trainer._global_step_counter[0]

        if trainer.model.training and trainer._aux_loss_fn is not None:
            if output_spt is not None and hasattr(output_spt, 'indices'):
                bsz = batch['input_fillers'].shape[0]
                warmup = min(1.0, step / 10000)
                aux = trainer._aux_loss_fn(output_spt, bsz, trainer._aux_d_filler)
                if torch.is_tensor(aux) and aux.requires_grad:
                    loss = loss + trainer._aux_lambda * warmup * aux
                    trainer._aux_history.append((step, aux.item()))

        if trainer.model.training:
            trainer._train_history.append((step, loss.item()))
            trainer._global_step_counter[0] += 1

        result[0] = loss
        return tuple(result)

    trainer.process_batch = patched_process_batch

    # --- Patch train_epoch: wrap forward pass with AMP autocast, use GradScaler ---
    if use_amp:
        import types, logging
        logger = logging.getLogger(__name__)
        original_train_epoch = trainer.train_epoch

        def amp_train_epoch(self):
            """AMP-wrapped train_epoch. Replaces forward+backward with autocast+scaler."""
            self.model.train()
            loss_accumulator = torch.tensor(0.0, device=self.device)
            correct_tokens_accumulator = torch.tensor(0.0, device=self.device)
            total_tokens_accumulator = 0
            correct_sequences_accumulator = torch.tensor(0.0, device=self.device)
            total_sequences_accumulator = 0
            cons_arg1_entropy_accumulator = 0
            cons_arg2_entropy_accumulator = 0
            accumulator_steps = 0
            lr = self.lr

            def reset_accumulators():
                nonlocal loss_accumulator, correct_tokens_accumulator, total_tokens_accumulator
                nonlocal correct_sequences_accumulator, total_sequences_accumulator
                nonlocal cons_arg1_entropy_accumulator, cons_arg2_entropy_accumulator, accumulator_steps
                loss_accumulator = torch.tensor(0.0, device=self.device)
                correct_tokens_accumulator = torch.tensor(0.0, device=self.device)
                total_tokens_accumulator = 0
                correct_sequences_accumulator = torch.tensor(0.0, device=self.device)
                total_sequences_accumulator = 0
                cons_arg1_entropy_accumulator = 0
                cons_arg2_entropy_accumulator = 0
                accumulator_steps = 0

            import time as _time
            start_time = _time.time()

            is_warmup = self.global_step < self.num_warmup_steps
            if is_warmup:
                lr = self.lr * self.global_step / max(1, self.num_warmup_steps)
                for param_group in self.optimizer.param_groups:
                    param_group['lr'] = lr

            for epoch_step, batch in enumerate(self.train_loader):
                is_warmup = self.global_step < self.num_warmup_steps
                if is_warmup:
                    lr = self.lr * self.global_step / max(1, self.num_warmup_steps)
                    for param_group in self.optimizer.param_groups:
                        param_group['lr'] = lr

                # === AMP FORWARD ===
                with torch.amp.autocast('cuda', enabled=self._amp_enabled):
                    batch_loss, batch_correct_tokens, batch_token_total, \
                        batch_correct_sequences, batch_total_sequences, _, \
                        batch_entropies, _, _ = self.process_batch(
                            batch, use_custom_memory=self.use_custom_memory)

                batch_correct_sequences = batch_correct_sequences.sum()

                if self.global_step < 10000:
                    current_entropy_coef = self.entropy_regularization_coefficient * (self.global_step / 10000)
                else:
                    current_entropy_coef = self.entropy_regularization_coefficient
                batch_loss += current_entropy_coef * (
                    batch_entropies['cons_arg1'].mean() + batch_entropies['cons_arg2'].mean())

                loss_accumulator += batch_loss.detach()
                correct_tokens_accumulator += batch_correct_tokens.detach()
                total_tokens_accumulator += batch_token_total
                correct_sequences_accumulator += batch_correct_sequences.detach()
                total_sequences_accumulator += batch_total_sequences
                cons_arg1_entropy_accumulator += batch_entropies['cons_arg1'].detach()
                cons_arg2_entropy_accumulator += batch_entropies['cons_arg2'].detach()
                accumulator_steps += 1

                # === AMP BACKWARD ===
                self._amp_scaler.scale(batch_loss).backward()
                if self.gclip > 0:
                    self._amp_scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.gclip)
                self._amp_scaler.step(self.optimizer)
                self._amp_scaler.update()
                self.optimizer.zero_grad(set_to_none=True)

                if self.scheduler and not is_warmup:
                    self.scheduler.step()
                    lr = self.optimizer.param_groups[0]['lr']

                if self.train_log_freq != -1 and self.global_step % self.train_log_freq == 0 and self.main_process:
                    train_acc = correct_sequences_accumulator / total_sequences_accumulator
                    train_partial_acc = correct_tokens_accumulator / total_tokens_accumulator
                    train_loss = loss_accumulator / accumulator_steps
                    cons_arg1_entropy = cons_arg1_entropy_accumulator / accumulator_steps
                    cons_arg2_entropy = cons_arg2_entropy_accumulator / accumulator_steps
                    dt = _time.time() - start_time
                    start_time = _time.time()
                    logger.info(f'Epoch {self.epoch}, step {epoch_step}/{len(self.train_loader)}, '
                                f'train_acc: {train_acc:.2f}, train_partial_acc: {train_partial_acc:.2f}, '
                                f'train_loss: {train_loss:.3f}, time: {dt:.2f} s')
                    cons_arg1_dict = {f'Train arg 1 step {idx}': value.item() for idx, value in enumerate(cons_arg1_entropy)}
                    cons_arg2_dict = {f'Train arg 2 step {idx}': value.item() for idx, value in enumerate(cons_arg2_entropy)}
                    wandb.log(dict(
                        {**cons_arg1_dict, **cons_arg2_dict},
                        epoch=self.epoch,
                        train_acc=train_acc,
                        train_partial_acc=train_partial_acc,
                        train_loss=train_loss,
                        lr=lr,
                    ), step=self.global_step)
                    reset_accumulators()

                self.global_step += 1
                if self.global_step >= self.num_steps:
                    break

            self.epoch += 1
            return (correct_sequences_accumulator / max(total_sequences_accumulator, 1),
                    correct_tokens_accumulator / max(total_tokens_accumulator, 1),
                    loss_accumulator / max(accumulator_steps, 1),
                    lr)

        trainer.train_epoch = types.MethodType(amp_train_epoch, trainer)
        print(f'  AMP enabled (GradScaler active)')

    return trainer


print('Monkey-patch defined (aux loss + AMP)')

In [ ]:
# Cell 7: Create experiment
#
# Replicates main.py setup for the active_logical task.
# Optimizations: batch_size=32, torch.compile on NTA (if available).

def create_experiment(condition, seed=42, aux_lambda=0.1, steps=None):
    """Create a fully configured trainer for one experimental condition.

    Args:
        condition: 'baseline', 'coprod', or 'random'
        seed: random seed
        aux_lambda: weight for auxiliary loss
        steps: training steps (defaults based on QUICK_MODE)
    Returns:
        trainer: patched Trainer ready for .train()
        data_loaders: dict of DataLoaders
    """
    if steps is None:
        steps = 2000 if QUICK_MODE else 20000

    args_list = [
        '--data_dir', 'data/active_logical',
        '--dtm_layers', '16',
        '--batch_size', '32',  # increased from 16 for better GPU util
        '--max_tree_depth', '13',
        '--sparse', '1',
        '--learn_filler_embed', '0',
        '--use_vocab_info', '1',
        '--tied_io_languages', '1',
        '--max_filled_roles', '1028',
        '--ctrl_hidden_dim', '64',
        '--lr', '1e-4',
        '--optim_beta2', '0.95',
        '--optim_beta1', '0.9',
        '--gclip', '1',
        '--wd', '1e-1',
        '--transformer_nheads', '4',
        '--router_dropout', '0.1',
        '--num_warmup_steps', '10000',
        '--scheduler', 'cosine',
        '--steps', str(steps),
        '--seed', str(seed),
        '--train_log_freq', '20',
        '--validate_every_num_epochs', '5',
        '--num_workers', '2',  # match typical Colab CPU count
    ]

    args = parse_args(args_list)

    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    random.seed(args.seed)
    torch.manual_seed(args.seed)

    # Data
    data_loaders, input_lang, output_lang = data.prepare_data_loaders(
        args.data_dir, args.max_tree_depth, args.add_eob_tokens, False,
        args.batch_size, args.num_workers,
        data_filter=args.data_filter,
        max_train_examples=args.max_train_examples,
        output_lowercase=args.output_lowercase,
        add_eob_to_memory=args.add_eob_to_memory,
        num_extra_tokens_in_memory=args.num_extra_tokens_in_memory,
    )

    # Merge vocabularies (same as main.py)
    input_vocab = set(input_lang.ind2vocab.values())
    output_vocab = set(output_lang.ind2vocab.values())
    input_unique_vocab = input_vocab - output_vocab
    for i, v in output_lang.ind2vocab.items():
        input_lang.add_word(v)
    output_lang = input_lang
    for loader in data_loaders.values():
        if loader:
            loader.dataset.output_lang = output_lang

    output_indices_mask = [0]
    if not args.tied_io_languages:
        for i, v in output_lang.ind2vocab.items():
            if v in input_unique_vocab:
                output_indices_mask.append(i)

    max_input_length = max(
        loader.dataset.max_input_length
        for loader in data_loaders.values() if loader
    )

    if not args.d_filler:
        args.d_filler = len(input_lang.ind2vocab)

    if args.steps is not None:
        args.epoch = math.ceil(args.steps / len(data_loaders['train']))

    if args.use_vocab_info:
        vocab_info = data.get_vocab_info(args.data_dir, output_lang.ind2vocab.values())
    else:
        vocab_info = {'unary': (), 'binary': (), 'terminal': ('<EOB>',)}

    # TPR
    tpr = TPR(
        args,
        num_input_fillers=len(input_lang.ind2vocab),
        num_output_fillers=len(output_lang.ind2vocab),
        num_roles=2 ** args.max_tree_depth,
        d_filler=args.d_filler,
        d_role=args.d_role,
        filler_emb_gain=args.filler_emb_gain,
        learn_empty_filler=args.learn_empty_filler,
        tied_io_languages=args.tied_io_languages,
        empty_filler_initialization=args.empty_filler_initialization,
        device=device,
        sparse=args.sparse,
        nt_token_index=output_lang.vocab2ind.get('<NT>', None),
    ).to(device=device)

    # Convert args (needed by DiffTreeMachine)
    from main import convert_args_to_config, setup_optimizer_and_scheduler
    hardcode_cons_root_index = None
    if args.hardcode_cons_root_token:
        if args.hardcode_cons_root_token == '-1':
            hardcode_cons_root_index = -1
        else:
            hardcode_cons_root_index = output_lang.vocab2ind[args.hardcode_cons_root_token]

    convert_args_to_config(args, input_lang, output_lang, tpr,
                           hardcode_cons_root_index, max_input_length)

    # Model
    dtm = DiffTreeMachine(args).to(device=device)

    # Try torch.compile on the NTA (transformer) submodule
    if torch.cuda.is_available() and hasattr(dtm, 'ctrl_net'):
        try:
            dtm.ctrl_net = torch.compile(dtm.ctrl_net, mode='reduce-overhead')
            print(f'  torch.compile applied to ctrl_net')
        except Exception as e:
            print(f'  torch.compile skipped: {e}')

    # Optimizer
    optimizer, scheduler = setup_optimizer_and_scheduler(dtm, args)

    # Trainer
    trainer = Trainer(
        dtm, tpr, data_loaders, optimizer,
        args.epoch, args.steps, args.num_warmup_steps,
        True,  # main_process
        False,  # is_ddp
        decoded_tpr_to_tree_fn(args.tpr_loss_type, sparse=args.sparse,
                               output_indices_mask=output_indices_mask),
        torch.nn.CrossEntropyLoss(),
        device,
        output_lang.ind2vocab,
        vocab_info,
        args.use_wandb,
        args.validate_every_num_epochs,
        args.train_log_freq,
        early_stop_epochs=args.early_stop_epochs,
        pad_idx=0,
        sparse=args.sparse,
        scheduler=scheduler,
        gclip=args.gclip,
        lr=args.lr,
        out_dir=args.out_dir,
        best_checkpoint_file=args.best_checkpoint_file,
        most_recent_checkpoint_file=args.most_recent_checkpoint_file,
        use_custom_memory=args.custom_memory,
        cross_entropy_weighting=args.cross_entropy_weighting,
        entropy_regularization_coefficient=args.entropy_regularization_coefficient,
        max_input_length=max_input_length,
        nt_token_index=output_lang.vocab2ind.get('<NT>', None),
        eob_token_index=output_lang.vocab2ind.get('<EOB>', None),
        output_indices_mask=output_indices_mask,
    )

    # Set up auxiliary loss based on condition
    d_filler = args.d_filler
    if condition == 'coprod':
        aux_fn = coassociativity_loss
    elif condition == 'random':
        aux_fn = RandomProjectionLoss(d_filler).to(device)
    else:
        aux_fn = None

    trainer = patch_trainer(trainer, aux_fn, aux_lambda, d_filler)

    return trainer, data_loaders


print('create_experiment defined')

In [ ]:
# Cell 8: Run 3 experiments

all_results = {}
all_histories = {}

for condition in ['baseline', 'coprod', 'random']:
    print(f"\n{'=' * 60}")
    print(f"Running condition: {condition}")
    print(f"{'=' * 60}")

    t0 = time.time()
    trainer, data_loaders = create_experiment(
        condition=condition, seed=42, aux_lambda=0.1
    )
    trainer.train()

    # Evaluate on all splits
    # test() returns (loss, partial_acc, full_acc, entropies, perplexity)
    results = {}
    for split_name in ['test', 'eval_new', 'eval_long']:
        loader = data_loaders.get(split_name)
        if loader:
            _, _, full_acc, _, _ = trainer.test(loader)
            results[f'{split_name}_acc'] = full_acc
            print(f"  {split_name}: {full_acc:.4f}")

    all_results[condition] = results
    all_histories[condition] = {
        'train': trainer._train_history,
        'aux': trainer._aux_history,
    }
    elapsed = time.time() - t0
    print(f"  Elapsed: {elapsed:.0f}s")

print("\nAll experiments complete.")

In [ ]:
# Cell 9: Plot training curves

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Training loss
for cond in ['baseline', 'coprod', 'random']:
    hist = all_histories[cond]['train']
    if hist:
        steps, losses = zip(*hist)
        # Smooth with running average
        window = max(1, len(losses) // 100)
        smoothed = [sum(losses[max(0,i-window):i+1]) / len(losses[max(0,i-window):i+1])
                    for i in range(len(losses))]
        axes[0].plot(steps, smoothed, label=cond, alpha=0.8)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].legend()

# Plot 2: Auxiliary loss (coprod vs random)
for cond in ['coprod', 'random']:
    hist = all_histories[cond]['aux']
    if hist:
        steps, losses = zip(*hist)
        window = max(1, len(losses) // 100)
        smoothed = [sum(losses[max(0,i-window):i+1]) / len(losses[max(0,i-window):i+1])
                    for i in range(len(losses))]
        axes[1].plot(steps, smoothed, label=cond, alpha=0.8)
axes[1].set_title('Auxiliary Loss')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Aux Loss')
axes[1].legend()

# Plot 3: Summary bar chart of test/OOD accuracies
splits = ['test_acc', 'eval_new_acc', 'eval_long_acc']
x = range(len(splits))
width = 0.25
for i, cond in enumerate(['baseline', 'coprod', 'random']):
    vals = [all_results[cond].get(s, 0) for s in splits]
    axes[2].bar([xi + i * width for xi in x], vals, width, label=cond)
axes[2].set_title('Accuracy by Split')
axes[2].set_xticks([xi + width for xi in x])
axes[2].set_xticklabels(['Test', 'OOD-New', 'OOD-Long'])
axes[2].set_ylim(0, 1)
axes[2].legend()

plt.tight_layout()
plt.savefig('hopf_poc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved hopf_poc_curves.png')

In [ ]:
# Cell 10: Summary table

print()
print('=' * 70)
print('RESULTS SUMMARY: Hopf Coassociativity PoC')
print('=' * 70)
print(f"{'Condition':<15} {'Test Acc':<12} {'OOD-New':<12} {'OOD-Long':<12}")
print('-' * 51)
for cond in ['baseline', 'coprod', 'random']:
    r = all_results[cond]
    test = r.get('test_acc', float('nan'))
    new = r.get('eval_new_acc', float('nan'))
    long = r.get('eval_long_acc', float('nan'))
    print(f'{cond:<15} {test:<12.4f} {new:<12.4f} {long:<12.4f}')
print('-' * 51)
print()
print('Interpretation guide:')
print('  coprod > random > baseline  =>  Algebraic structure matters')
print('  coprod ~ random > baseline  =>  Just regularization effect')
print('  all ~ equal                 =>  Neither helps on this task')
print()
print(f'Mode: {"QUICK (2k steps)" if QUICK_MODE else "FULL (20k steps)"}')
print('Note: meaningful conclusions require FULL mode + multiple seeds.')